# Exceptions

> Custom exception classes with notebook location tracking

In [ ]:
#| default_exp utils.exceptions

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Optional, Any
from dataclasses import dataclass
from pathlib import Path
import warnings

## Notebook Location

Track where an error/warning originated in the source notebooks.

In [ ]:
#| export
@dataclass
class NotebookLocation:
    """Track the source location of an error within nbdev notebooks.
    
    Args:
        notebook: Name or path of the source notebook (e.g., '03_utils.ipynb').
        cell: Zero-based cell index where the error occurred.
        line: One-based line number within the cell (default 0 = unknown).
    """
    notebook: str
    cell: int = -1
    line: int = 0
    
    def __str__(self) -> str:
        """Format location as 'notebook:cell[N]:line[M]'."""
        parts = [self.notebook]
        if self.cell >= 0:
            parts.append(f'cell[{self.cell}]')
        if self.line > 0:
            parts.append(f'line[{self.line}]')
        return ':'.join(parts)
    
    @classmethod
    def from_path(cls, path: Path, cell: int = -1, line: int = 0) -> 'NotebookLocation':
        """Create a location from a Path object.
        
        Args:
            path: Path to the notebook file.
            cell: Zero-based cell index.
            line: One-based line number within the cell.
        
        Returns:
            A new NotebookLocation instance.
        """
        return cls(notebook=path.name, cell=cell, line=line)

## Base Exception

In [ ]:
#| export
class NotebookException(Exception):
    """Base exception for nbdev-mcp with optional notebook location tracking.
    
    Args:
        message: The error message.
        location: Optional source location in the notebooks where the error originated.
    """
    def __init__(self, message: str, location: Optional[NotebookLocation] = None):
        self.location = location
        if location:
            message = f'{message} [{location}]'
        super().__init__(message)

## Specific Exceptions

In [ ]:
#| export
class ProjectNotFoundError(NotebookException):
    """Raised when an nbdev project cannot be found or resolved.
    
    Args:
        selector: The selector/path that failed to resolve.
    """
    def __init__(self, selector: str):
        super().__init__(f"Project not found: '{selector}'")
        self.selector = selector

In [ ]:
#| export
class NotebookNotFoundError(NotebookException):
    """Raised when a notebook file cannot be found.
    
    Args:
        path: The notebook path that was not found.
    """
    def __init__(self, path: Any):
        super().__init__(f"Notebook not found: '{path}'")
        self.path = path

In [ ]:
#| export
class CellNotFoundError(NotebookException):
    """Raised when a cell index is out of range.
    
    Args:
        notebook: The notebook name.
        cell_index: The invalid cell index.
        total_cells: The actual number of cells in the notebook.
    """
    def __init__(self, notebook: str, cell_index: int, total_cells: int):
        loc = NotebookLocation(notebook, cell=cell_index)
        super().__init__(
            f"Cell index {cell_index} out of range (notebook has {total_cells} cells)",
            location=loc
        )
        self.cell_index = cell_index
        self.total_cells = total_cells

In [ ]:
#| export
class ExportError(NotebookException):
    """Raised when nbdev_export fails.
    
    Args:
        message: Error details from the export command.
        location: Source location if known.
    """
    pass

In [ ]:
#| export
class LintError(NotebookException):
    """Raised when linting finds a blocking issue.
    
    Args:
        rule: The lint rule that was violated.
        message: Description of the violation.
        location: Source location of the violation.
    """
    def __init__(self, rule: str, message: str, location: Optional[NotebookLocation] = None):
        super().__init__(f"[{rule}] {message}", location=location)
        self.rule = rule

## Warnings

In [ ]:
#| export
class NbdevMcpWarning(UserWarning):
    """Base warning for nbdev-mcp with optional notebook location.
    
    Args:
        message: The warning message.
        location: Source location in the notebooks.
    """
    def __init__(self, message: str, location: Optional[NotebookLocation] = None):
        self.location = location
        if location:
            message = f'{message} [{location}]'
        super().__init__(message)

In [ ]:
#| export
class DeprecationWarningNbdev(NbdevMcpWarning):
    """Warning for deprecated nbdev-mcp features."""
    pass


class StyleWarning(NbdevMcpWarning):
    """Warning for style/convention violations that aren't errors."""
    pass

## Helper Functions

In [ ]:
#| export
def warn_with_location(
    message: str,
    notebook: Optional[str] = None,
    cell: int = -1,
    line: int = 0,
    category: type = NbdevMcpWarning
) -> None:
    """Issue a warning with notebook location context.
    
    Args:
        message: The warning message.
        notebook: Name of the source notebook.
        cell: Zero-based cell index (default -1 = unknown).
        line: One-based line number within cell (default 0 = unknown).
        category: Warning category class (default NbdevMcpWarning).
    """
    loc = NotebookLocation(notebook, cell, line) if notebook else None
    warnings.warn(category(message, location=loc), stacklevel=2)

In [ ]:
#| export
def raise_with_location(
    message: str,
    notebook: Optional[str] = None,
    cell: int = -1,
    line: int = 0,
    exc_class: type = NotebookException
) -> None:
    """Raise an exception with notebook location context.
    
    Args:
        message: The error message.
        notebook: Name of the source notebook.
        cell: Zero-based cell index (default -1 = unknown).
        line: One-based line number within cell (default 0 = unknown).
        exc_class: Exception class to raise (default NotebookException).
    
    Raises:
        NotebookException: Or subclass specified by exc_class.
    """
    loc = NotebookLocation(notebook, cell, line) if notebook else None
    raise exc_class(message, location=loc)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()